# E-commerce Data Analysis using PySpark

#### by Pramod Godse

In [38]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import broadcast
import builtins
from pyspark.sql.functions import col
import pandas as pd
from pyspark.sql.functions import count, countDistinct, sum, avg, desc, col

spark = SparkSession.builder \
    .appName("Ecommerce PySpark Assignment") \
    .getOrCreate()

### Read CSV files into Spark DataFrames and inspect schema, sample rows, and row counts.

In [2]:
path = "D:/data Engineering/Asignment/PySpark Assignment/Dataset/"

users_df = spark.read.csv(path + "users.csv", header=True, inferSchema=True)
orders_df = spark.read.csv(path + "orders.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv(path + "order_items.csv", header=True, inferSchema=True)
products_df = spark.read.csv(path + "products.csv", header=True, inferSchema=True)

In [5]:
import pandas as pd
from pyspark.sql.functions import col

dataframes = {
    "users": users_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "products": products_df
}

def inspect_df(df, name):
    print(f"\n{'='*25} {name.upper()} {'='*25}")
    
    print("\nSample Data:")
    display(df.limit(5).toPandas())
    
    print("\nSchema:")
    schema_data = [
        {
            "Column": f.name,
            "Data Type": f.dataType.simpleString(),
            "Nullable": f.nullable
        }
        for f in df.schema.fields
    ]
    display(pd.DataFrame(schema_data))
    
    print("\nRow Count:")
    print(df.count())
    
    print("\nMissing Values in Dataset:")
    missing_data = pd.DataFrame({
        "Column": df.columns,
        "Missing_Count": [
            df.filter(col(c).isNull()).count()
            for c in df.columns
        ]
    })
    
    missing_data = missing_data[missing_data["Missing_Count"] > 0] \
        .sort_values("Missing_Count", ascending=False)
    
    if missing_data.empty:
        print("No missing values found.")
    else:
        display(missing_data)

for name, df in dataframes.items():
    inspect_df(df, name)


========================= USERS =========================

Sample Data:


,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at
0,44262,Michael,Sanchez,michaelsanchez@example.net,48,M,Mie,5379 Kim Corner,513-0836,Suzuka City,Japan,34.851814,136.508713,Facebook,2020-12-05 20:09:00
1,61852,David,Watson,davidwatson@example.org,21,M,Mie,58568 Brooks Plain Apt. 269,513-0836,Suzuka City,Japan,34.851814,136.508713,Search,2022-01-24 18:30:00
2,82418,Lisa,Rivera,lisarivera@example.net,29,F,Mie,3092 Perez Overpass,513-0836,Suzuka City,Japan,34.851814,136.508713,Search,2019-09-07 12:59:00
3,23274,Logan,Flores,loganflores@example.com,53,M,Acre,412 Underwood Tunnel Suite 025,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Search,2020-06-28 20:09:00
4,30022,Kathy,Peterson,kathypeterson@example.net,23,F,Acre,8480 Alexandra Village,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Email,2021-06-01 13:00:00



Schema:


,Column,Data Type,Nullable
0,id,int,True
1,first_name,string,True
2,last_name,string,True
3,email,string,True
4,age,int,True
5,gender,string,True
6,state,string,True
7,street_address,string,True
8,postal_code,string,True
9,city,string,True



Row Count:
100000

Missing Values in Dataset:
No missing values found.

========================= ORDERS =========================

Sample Data:


...

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,27569,92.652563,Swim,2XU Men's Swimmers Compression Long Sleeve Top,2XU,150.410004,Men,B23C5765E165D83AA924FA8F13C05F25,1
1,27445,24.719661,Swim,TYR Sport Men's Square Leg Short Swim Suit,TYR,38.990002,Men,2AB7D3B23574C3DEA2BD278AFD0939AB,1
2,27457,15.897600,Swim,TYR Sport Men's Solid Durafast Jammer Swim Suit,TYR,27.600000,Men,8F831227B0EB6C6D09A0555531365933,1
3,27466,17.850000,Swim,TYR Sport Men's Swim Short/Resistance Short Sw...,TYR,30.000000,Men,67317D6DCC4CB778AEB9219565F5456B,1
4,27481,29.408001,Swim,TYR Alliance Team Splice Jammer,TYR,45.950001,Men,213C888198806EF1A0E2BBF2F4855C6C,1



Schema:


,Column,Data Type,Nullable
0,id,int,True
1,cost,double,True
2,category,string,True
3,name,string,True
4,brand,string,True
5,retail_price,double,True
6,department,string,True
7,sku,string,True
8,distribution_center_id,int,True



Row Count:
29120

Missing Values in Dataset:


,Column,Missing_Count
4,brand,24
3,name,2


### Clean essential columns and handle null values in join keys and descriptive fields.

In [6]:
# Clean null values in join keys
users_clean = users_df.dropna(subset=["id"])

orders_clean = orders_df.dropna(subset=["order_id", "user_id"])

order_items_clean = order_items_df.dropna(subset=["order_id", "product_id"])

products_clean = products_df.dropna(subset=["id"])


In [7]:
# Clean descriptive fields
products_clean = products_clean.fillna({
    "brand": "unknown",
    "name": "unknown",
    "category": "unknown"
})

In [8]:
# datatype of Numeric columns

order_items_clean = order_items_clean.withColumn(
    "sale_price",
    col("sale_price").cast("double")
)

products_clean = products_clean.withColumn(
    "retail_price",
    col("retail_price").cast("double")
).withColumn(
    "cost",
    col("cost").cast("double")
)

### Transformations

In [9]:
from pyspark.sql.functions import col, when, count, countDistinct

# Select required columns from order_items
order_items_selected = order_items_clean.select(
    "order_id",
    "user_id",
    "product_id",
    "status",
    "sale_price",
    "created_at"
)

order_items_selected.show(5)

+--------+-------+----------+-------+----------+-------------------+
|order_id|user_id|product_id| status|sale_price|         created_at|
+--------+-------+----------+-------+----------+-------------------+
|   10826|   8614|     13606|Shipped|       2.5|2020-06-28 06:10:17|
|   13243|  10505|     13606|Shipped|       2.5|2022-03-01 10:48:44|
|   53140|  42340|     13606|Shipped|       2.5|2021-04-11 07:01:52|
|  104681|  83704|     13606|Shipped|       2.5|2022-03-30 03:07:06|
|  117931|  94363|     13606|Shipped|       2.5|2021-02-17 14:58:17|
+--------+-------+----------+-------+----------+-------------------+
only showing top 5 rows


In [10]:
# Create price_bucket column
order_items_transformed = order_items_selected.withColumn(
    "price_bucket",
    when(col("sale_price") < 25, "Low")
    .when((col("sale_price") >= 25) & (col("sale_price") <= 75), "Medium")
    .otherwise("High")
)

order_items_transformed.show(5)

+--------+-------+----------+-------+----------+-------------------+------------+
|order_id|user_id|product_id| status|sale_price|         created_at|price_bucket|
+--------+-------+----------+-------+----------+-------------------+------------+
|   10826|   8614|     13606|Shipped|       2.5|2020-06-28 06:10:17|         Low|
|   13243|  10505|     13606|Shipped|       2.5|2022-03-01 10:48:44|         Low|
|   53140|  42340|     13606|Shipped|       2.5|2021-04-11 07:01:52|         Low|
|  104681|  83704|     13606|Shipped|       2.5|2022-03-30 03:07:06|         Low|
|  117931|  94363|     13606|Shipped|       2.5|2021-02-17 14:58:17|         Low|
+--------+-------+----------+-------+----------+-------------------+------------+
only showing top 5 rows


In [11]:
# Filter records
order_items_filtered = order_items_transformed.filter(
    (col("sale_price") > 20) & (col("product_id").isNotNull())
)

order_items_filtered.show(5)

+--------+-------+----------+-------+----------+-------------------+------------+
|order_id|user_id|product_id| status|sale_price|         created_at|price_bucket|
+--------+-------+----------+-------+----------+-------------------+------------+
|    1555|   1248|     12343|Shipped|      21.0|2022-01-23 21:50:05|         Low|
|    2003|   1603|     26412|Shipped|      21.0|2021-01-03 05:08:11|         Low|
|    2163|   1743|     24521|Shipped|      21.0|2022-04-05 03:35:08|         Low|
|    2209|   1777|     24787|Shipped|      21.0|2020-12-08 07:56:58|         Low|
|    2421|   1944|     24521|Shipped|      21.0|2021-01-19 14:52:24|         Low|
+--------+-------+----------+-------+----------+-------------------+------------+
only showing top 5 rows


In [12]:
# Summary counts
order_items_filtered.agg(
    count("*").alias("total_rows"),
    countDistinct("order_id").alias("unique_orders"),
    countDistinct("user_id").alias("unique_users"),
    countDistinct("product_id").alias("unique_products")
).show()

+----------+-------------+------------+---------------+
|total_rows|unique_orders|unique_users|unique_products|
+----------+-------------+------------+---------------+
|    144100|       106092|       71873|          23044|
+----------+-------------+------------+---------------+



### Joins

In [13]:
# Join users with orders

users_orders_df = users_clean.join(
    orders_clean,
    users_clean.id == orders_clean.user_id,
    "inner"
)

#users_orders_df.show(5)
display(users_orders_df.limit(5).toPandas())

,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,...,created_at,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,44262,Michael,Sanchez,michaelsanchez@example.net,48,M,Mie,5379 Kim Corner,513-0836,Suzuka City,...,2020-12-05 20:09:00,55564,44262,Cancelled,Male,2021-11-15 20:09:00,NaT,NaT,NaT,1
1,61852,David,Watson,davidwatson@example.org,21,M,Mie,58568 Brooks Plain Apt. 269,513-0836,Suzuka City,...,2022-01-24 18:30:00,77700,61852,Complete,Male,2022-05-19 18:30:00,NaT,2022-05-22 01:34:00,2022-05-27 01:09:00,1
2,82418,Lisa,Rivera,lisarivera@example.net,29,F,Mie,3092 Perez Overpass,513-0836,Suzuka City,...,2019-09-07 12:59:00,103151,82418,Shipped,Female,2021-06-16 12:59:00,NaT,2021-06-17 20:45:00,NaT,1
3,82418,Lisa,Rivera,lisarivera@example.net,29,F,Mie,3092 Perez Overpass,513-0836,Suzuka City,...,2019-09-07 12:59:00,103150,82418,Shipped,Female,2019-11-16 12:59:00,NaT,2019-11-18 12:33:00,NaT,2
4,23274,Logan,Flores,loganflores@example.com,53,M,Acre,412 Underwood Tunnel Suite 025,69917-400,Rio Branco,...,2020-06-28 20:09:00,29199,23274,Cancelled,Male,2021-05-06 20:09:00,NaT,NaT,NaT,1


In [14]:
# Join orders with order_items

orders_items_df = users_orders_df.join(
    order_items_filtered,
    orders_clean.order_id == order_items_filtered.order_id,
    "inner"
)

#orders_items_df.show(5)
display(orders_items_df.limit(5).toPandas())

,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,...,shipped_at,delivered_at,num_of_item,order_id,user_id,product_id,status,sale_price,created_at,price_bucket
0,24,Reginald,Huffman,reginaldhuffman@example.org,64,M,Comunidad Valenciana,9284 Mahoney Islands,46100,Burjasot,...,NaT,NaT,2,31,24,26061,Cancelled,30.000000,2022-06-04 18:51:08.000000,Medium
1,24,Reginald,Huffman,reginaldhuffman@example.org,64,M,Comunidad Valenciana,9284 Mahoney Islands,46100,Burjasot,...,NaT,NaT,2,31,24,22183,Cancelled,59.500000,2022-05-31 17:12:31.000000,Medium
2,27,Jeanette,White,jeanettewhite@example.net,43,F,Texas,79040 Tyler Field,78613,Cedar Park,...,NaT,NaT,1,34,27,6010,Processing,39.240002,2022-05-03 12:49:28.000000,Medium
3,58,Denise,Noble,denisenoble@example.net,62,F,Guangdong,8396 George Route Suite 754,510475,Tianjin,...,2022-06-04 19:53:14.329487,NaT,3,78,58,2602,Shipped,25.000000,2022-06-06 08:37:33.329487,Medium
4,64,Brandon,Singleton,brandonsingleton@example.net,52,M,Jilin,79524 Charles Track,136199,Shanghai,...,2022-04-20 08:37:00.000000,2022-04-23 17:22:00,1,85,64,23912,Complete,109.989998,2022-04-19 08:48:48.000000,High


In [15]:
# Join with products and create final_df

final_df = orders_items_df.join(
    broadcast(products_clean),
    order_items_filtered.product_id == products_clean.id,
    "inner"
)

In [16]:
# Show schema, row count, and 10 records
u = users_clean.alias("u")
o = orders_clean.alias("o")
oi = order_items_filtered.alias("oi")
p = products_clean.alias("p")

final_df = u.join(
    o,
    col("u.id") == col("o.user_id"),
    "inner"
).join(
    oi,
    col("o.order_id") == col("oi.order_id"),
    "inner"
).join(
    broadcast(p),
    col("oi.product_id") == col("p.id"),
    "inner"
).select(
    col("u.id").alias("customer_id"),
    col("u.gender"),
    col("u.state"),
    col("u.country"),
    col("o.order_id").alias("order_id"),
    col("o.status").alias("order_status"),
    col("oi.product_id"),
    col("oi.status").alias("item_status"),
    col("oi.sale_price"),
    col("oi.created_at").alias("item_created_at"),
    col("oi.price_bucket"),
    col("p.name").alias("product_name"),
    col("p.category"),
    col("p.brand"),
    col("p.retail_price"),
    col("p.cost")
)

final_df.printSchema()
print("Final Row Count:", final_df.count())

print("Sample Records:")
#final_df.show(10, truncate=False)
display(final_df.limit(10).toPandas())

root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- item_status: string (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- item_created_at: timestamp (nullable = true)
 |-- price_bucket: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- category: string (nullable = false)
 |-- brand: string (nullable = false)
 |-- retail_price: double (nullable = true)
 |-- cost: double (nullable = true)

Final Row Count: 144100
Sample Records:


,customer_id,gender,state,country,order_id,order_status,product_id,item_status,sale_price,item_created_at,price_bucket,product_name,category,brand,retail_price,cost
0,7,M,Seoul,South Korea,12,Cancelled,21516,Cancelled,75.000000,2022-05-06 12:32:06.000000,Medium,Dylan George Signature Boot Cut,Jeans,Dylan George,75.000000,36.150000
1,18,M,California,United States,26,Shipped,23647,Shipped,74.000000,2021-04-28 09:06:54.000000,Medium,Toes on the Nose Men's Shadows Fleece,Outerwear & Coats,Toes on the Nose,74.000000,34.854000
2,24,M,Comunidad Valenciana,Spain,31,Cancelled,26061,Cancelled,30.000000,2022-06-04 18:51:08.000000,Medium,Michael Kors Men's 3 Pack Shirt,Underwear,Michael Kors,30.000000,12.810000
3,24,M,Comunidad Valenciana,Spain,31,Cancelled,22183,Cancelled,59.500000,2022-05-31 17:12:31.000000,Medium,Quiksilver Young Men's Dane 2 New Slim Tapered...,Pants,Quiksilver,59.500000,25.882500
4,27,F,Texas,United States,34,Processing,6010,Processing,39.240002,2022-05-03 12:49:28.000000,Medium,DKNYC Women's Colorblock Legging,Leggings,DKNYC,39.240002,23.700961
5,57,M,Shanghai,China,76,Complete,17351,Complete,59.500000,2020-01-22 20:19:29.000000,Medium,ecko unltd. Men's Eu72 Vertical Logo Hoodie Sw...,Fashion Hoodies & Sweatshirts,Ecko Unltd.,59.500000,31.654000
6,57,M,Shanghai,China,76,Complete,26764,Complete,59.500000,2020-01-26 23:31:34.000000,Medium,American Essentials Men's Ultra Soft Jersey Lo...,Sleep & Lounge,American Essentials,59.500000,22.312500
7,57,M,Shanghai,China,76,Complete,20840,Complete,325.950012,2020-01-25 21:04:52.000000,High,True Religion Men's Ricky Straight Jean,Jeans,True Religion,325.950012,184.813657
8,58,F,Guangdong,China,78,Shipped,2602,Shipped,25.000000,2022-06-06 08:37:33.329487,Medium,Spalding Women's Ankle Legging,Active,Spalding,25.000000,10.250000
9,62,M,Ceará,Brasil,81,Processing,22493,Processing,68.000000,2020-04-21 15:18:14.000000,Medium,American Apparel Cotton Edam Canvas Travel Pant,Pants,American Apparel,68.000000,32.844000


### Aggregations

In [17]:
# Top 10 product categories by number of items sold

top_categories_items = final_df.groupBy("category") \
    .agg(count("*").alias("items_sold")) \
    .orderBy(desc("items_sold"))

top_categories_items.show(10, truncate=False)

+-----------------------------+----------+
|category                     |items_sold|
+-----------------------------+----------+
|Jeans                        |12440     |
|Fashion Hoodies & Sweatshirts|10984     |
|Swim                         |10722     |
|Sweaters                     |10266     |
|Tops & Tees                  |9555      |
|Sleep & Lounge               |9532      |
|Shorts                       |9463      |
|Outerwear & Coats            |8929      |
|Intimates                    |8785      |
|Active                       |6916      |
+-----------------------------+----------+
only showing top 10 rows


In [18]:
# Top 10 brands by total sales
top_brands_sales = final_df.groupBy("brand") \
    .agg(sum("sale_price").alias("total_sales")) \
    .orderBy(desc("total_sales"))

top_brands_sales.show(10, truncate=False)

+-----------------+------------------+
|brand            |total_sales       |
+-----------------+------------------+
|Diesel           |196999.49990463257|
|Calvin Klein     |194493.8803539276 |
|True Religion    |177362.90949249268|
|Carhartt         |172878.0888080597 |
|7 For All Mankind|169054.3104057312 |
|Tommy Hilfiger   |123265.66032028198|
|Volcom           |110722.89968109131|
|The North Face   |108887.59001541138|
|Joe's Jeans      |102971.69995689392|
|Columbia         |101259.97943687439|
+-----------------+------------------+
only showing top 10 rows


In [19]:
# Top 10 states by number of orders

top_states_orders = final_df.groupBy("state") \
    .agg(countDistinct("order_id").alias("order_count")) \
    .orderBy(desc("order_count"))

top_states_orders.show(10, truncate=False)

+----------+-----------+
|state     |order_count|
+----------+-----------+
|Guangdong |5668       |
|England   |4303       |
|California|3895       |
|Shanghai  |2623       |
|Texas     |2578       |
|São Paulo |2291       |
|Beijing   |2203       |
|Zhejiang  |2191       |
|Hebei     |2151       |
|Jiangsu   |1988       |
+----------+-----------+
only showing top 10 rows


In [20]:
# Top 10 states by total sales value

top_states_sales = final_df.groupBy("state") \
    .agg(sum("sale_price").alias("total_sales")) \
    .orderBy(desc("total_sales"))

top_states_sales.show(10, truncate=False)

+----------+------------------+
|state     |total_sales       |
+----------+------------------+
|Guangdong |542239.4109611511 |
|England   |413095.310464859  |
|California|377514.39033699036|
|Texas     |249549.10032463074|
|Shanghai  |248982.1503868103 |
|São Paulo |221655.76026916504|
|Beijing   |216997.66023254395|
|Hebei     |212549.43022155762|
|Zhejiang  |209744.1404762268 |
|Jiangsu   |184951.90028381348|
+----------+------------------+
only showing top 10 rows


In [21]:
# Average sale price by category

avg_sale_price_category = final_df.groupBy("category") \
    .agg(avg("sale_price").alias("avg_sale_price")) \
    .orderBy(desc("avg_sale_price"))

avg_sale_price_category.show(truncate=False)

+-----------------------------+------------------+
|category                     |avg_sale_price    |
+-----------------------------+------------------+
|Outerwear & Coats            |145.48417383506487|
|Suits & Sport Coats          |135.6095157266043 |
|Suits                        |120.71641374182153|
|Blazers & Jackets            |117.68177614595119|
|Dresses                      |100.35295115070683|
|Jeans                        |100.09620350595456|
|Clothing Sets                |90.79550560940517 |
|Sweaters                     |80.98807814654008 |
|Jumpsuits & Rompers          |72.75053079001796 |
|Accessories                  |64.96500362015583 |
|Pants                        |64.28213277934438 |
|Skirts                       |61.713274178985294|
|Pants & Capris               |60.87543337848067 |
|Active                       |60.6277328276648  |
|Plus                         |59.8928465462852  |
|Swim                         |59.73595793250336 |
|Fashion Hoodies & Sweatshirts|

In [22]:
# Count of completed, returned, and cancelled items

item_status_count = final_df.groupBy("item_status") \
    .agg(count("*").alias("item_count")) \
    .orderBy(desc("item_count"))

item_status_count.show(truncate=False)

+-----------+----------+
|item_status|item_count|
+-----------+----------+
|Shipped    |43588     |
|Complete   |35856     |
|Processing |28697     |
|Cancelled  |21449     |
|Returned   |14510     |
+-----------+----------+



### Business Questions.

In [23]:
# Which state generated the highest sales?
highest_sales_state = final_df.groupBy("state") \
    .agg(sum("sale_price").alias("total_sales")) \
    .orderBy(desc("total_sales"))

highest_sales_state.show(1, truncate=False)

+---------+-----------------+
|state    |total_sales      |
+---------+-----------------+
|Guangdong|542239.4109611511|
+---------+-----------------+
only showing top 1 row


In [24]:
# Which category generated the highest revenue?
highest_revenue_category = final_df.groupBy("category") \
    .agg(sum("sale_price").alias("revenue")) \
    .orderBy(desc("revenue"))

highest_revenue_category.show(1, truncate=False)

+-----------------+-----------------+
|category         |revenue          |
+-----------------+-----------------+
|Outerwear & Coats|1299028.188173294|
+-----------------+-----------------+
only showing top 1 row


In [25]:
# Which brand appears most frequently?
most_frequent_brand = final_df.groupBy("brand") \
    .agg(count("*").alias("frequency")) \
    .orderBy(desc("frequency"))

most_frequent_brand.show(1, truncate=False)

+------------+---------+
|brand       |frequency|
+------------+---------+
|Calvin Klein|2851     |
+------------+---------+
only showing top 1 row


In [26]:
# Which gender placed more orders?

gender_order_count = final_df.groupBy("gender") \
    .agg(countDistinct("order_id").alias("order_count")) \
    .orderBy(desc("order_count"))

gender_order_count.show(truncate=False)

+------+-----------+
|gender|order_count|
+------+-----------+
|M     |55361      |
|F     |50731      |
+------+-----------+



In [27]:
# Which orders contain more than one item?
multi_item_orders = final_df.groupBy("order_id") \
    .agg(count("*").alias("item_count")) \
    .filter(col("item_count") > 1) \
    .orderBy(desc("item_count"))

multi_item_orders.show(20, truncate=False)

+--------+----------+
|order_id|item_count|
+--------+----------+
|42304   |4         |
|25812   |4         |
|18502   |4         |
|6357    |4         |
|19131   |4         |
|31951   |4         |
|42613   |4         |
|46994   |4         |
|1265    |4         |
|21389   |4         |
|26965   |4         |
|463     |4         |
|40912   |4         |
|35820   |4         |
|41838   |4         |
|27466   |4         |
|45629   |4         |
|19396   |4         |
|21126   |4         |
|31035   |4         |
+--------+----------+
only showing top 20 rows


In [28]:
# Which categories have the highest average sale price?
highest_avg_price_categories = final_df.groupBy("category") \
    .agg(avg("sale_price").alias("avg_sale_price")) \
    .orderBy(desc("avg_sale_price"))

highest_avg_price_categories.show(10, truncate=False)

+-------------------+------------------+
|category           |avg_sale_price    |
+-------------------+------------------+
|Outerwear & Coats  |145.48417383506487|
|Suits & Sport Coats|135.6095157266043 |
|Suits              |120.71641374182153|
|Blazers & Jackets  |117.68177614595119|
|Dresses            |100.35295115070683|
|Jeans              |100.09620350595456|
|Clothing Sets      |90.79550560940517 |
|Sweaters           |80.98807814654008 |
|Jumpsuits & Rompers|72.75053079001796 |
|Accessories        |64.96500362015583 |
+-------------------+------------------+
only showing top 10 rows


### Cache / Persist

In [29]:
# Cache final_df
final_df.cache()

DataFrame[customer_id: int, gender: string, state: string, country: string, order_id: int, order_status: string, product_id: int, item_status: string, sale_price: double, item_created_at: timestamp, price_bucket: string, product_name: string, category: string, brand: string, retail_price: double, cost: double]

In [30]:
final_df.count()

144100

In [31]:
# Analysis 1: Category-wise revenue
category_revenue_cached = final_df.groupBy("category") \
    .agg(sum("sale_price").alias("revenue")) \
    .orderBy(desc("revenue"))

category_revenue_cached.show(10, truncate=False)

+-----------------------------+------------------+
|category                     |revenue           |
+-----------------------------+------------------+
|Outerwear & Coats            |1299028.188173294 |
|Jeans                        |1245196.7716140747|
|Sweaters                     |831423.6102523804 |
|Swim                         |640488.940952301  |
|Suits & Sport Coats          |638585.2095565796 |
|Fashion Hoodies & Sweatshirts|626209.2706165314 |
|Sleep & Lounge               |525929.1517467499 |
|Shorts                       |482181.7614841461 |
|Tops & Tees                  |457569.59177207947|
|Dresses                      |449581.2211551666 |
+-----------------------------+------------------+
only showing top 10 rows


In [32]:
# Analysis 2: State-wise order count
state_order_count_cached = final_df.groupBy("state") \
    .agg(countDistinct("order_id").alias("order_count")) \
    .orderBy(desc("order_count"))

state_order_count_cached.show(10, truncate=False)

+----------+-----------+
|state     |order_count|
+----------+-----------+
|Guangdong |5668       |
|England   |4303       |
|California|3895       |
|Shanghai  |2623       |
|Texas     |2578       |
|São Paulo |2291       |
|Beijing   |2203       |
|Zhejiang  |2191       |
|Hebei     |2151       |
|Jiangsu   |1988       |
+----------+-----------+
only showing top 10 rows


### Broadcast Join

In [33]:
# Apply broadcast join


u = users_clean.alias("u")
o = orders_clean.alias("o")
oi = order_items_filtered.alias("oi")
p = products_clean.alias("p")

final_df_broadcast = u.join(
    o,
    col("u.id") == col("o.user_id"),
    "inner"
).join(
    oi,
    col("o.order_id") == col("oi.order_id"),
    "inner"
).join(
    broadcast(p),
    col("oi.product_id") == col("p.id"),
    "inner"
).select(
    col("u.id").alias("customer_id"),
    col("u.gender"),
    col("u.state"),
    col("u.country"),
    col("o.order_id").alias("order_id"),
    col("o.status").alias("order_status"),
    col("oi.product_id"),
    col("oi.status").alias("item_status"),
    col("oi.sale_price"),
    col("oi.created_at").alias("item_created_at"),
    col("oi.price_bucket"),
    col("p.name").alias("product_name"),
    col("p.category"),
    col("p.brand"),
    col("p.retail_price"),
    col("p.cost")
)

#final_df_broadcast.show(10, truncate=False)
display(final_df_broadcast.limit(10).toPandas())
print("Broadcast Final DataFrame Count:", final_df_broadcast.count())


,customer_id,gender,state,country,order_id,order_status,product_id,item_status,sale_price,item_created_at,price_bucket,product_name,category,brand,retail_price,cost
0,113,F,Liaoning,China,148,Complete,13424,Complete,80.000000,2020-09-09 14:24:15,High,Luli Fama Cosita Buena Halter Top,Swim,Luli Fama,80.000000,34.400000
1,378,M,Piauí,Brasil,463,Complete,16217,Complete,64.949997,2022-05-19 08:37:02,Medium,Woolrich Men's Sportsman Chamois Shirt,Tops & Tees,Woolrich,64.949997,38.385448
2,378,M,Piauí,Brasil,463,Complete,21639,Complete,69.449997,2022-05-21 08:24:18,Medium,Volcom Slergo Denim Pant - Men's,Jeans,Volcom,69.449997,34.099948
3,378,M,Piauí,Brasil,463,Complete,17855,Complete,45.930000,2022-05-19 10:26:03,Medium,Zoo York Men's Grainger Sherpa Smolder Hoodie,Fashion Hoodies & Sweatshirts,ZOO YORK,45.930000,25.812660
4,378,M,Piauí,Brasil,463,Complete,16545,Complete,43.950001,2022-05-22 08:44:35,Medium,Port Authority Long Sleeve Easy Care Shirt (S6...,Tops & Tees,Port Authority,43.950001,26.106300
5,388,M,New Mexico,United States,471,Complete,17841,Complete,36.000000,2022-02-09 19:57:25,Medium,TapouT Men's Tri Skull Thermal Fashion Hoodie,Fashion Hoodies & Sweatshirts,TapouT,36.000000,20.124000
6,660,F,Hubei,China,833,Complete,1436,Complete,149.000000,2020-02-15 12:08:42,High,Lucky Brand Women's Lana Cowl Neck Poncho,Sweaters,Lucky Brand,149.000000,67.348000
7,858,F,Shanghai,China,1088,Processing,42,Processing,99.000000,2020-11-21 23:44:20,High,Lucky Brand Women's Plus-Size Farrah Printed Top,Tops & Tees,Lucky Brand,99.000000,51.480000
8,858,F,Shanghai,China,1088,Processing,4088,Processing,47.990002,2020-11-21 21:25:31,Medium,GREEN JUMPSUIT LACING POCKETS LAGENLOOK - FITS...,Jumpsuits & Rompers,LOTUSTRADERS,47.990002,24.762841
9,980,F,Gyeonggi-do,South Korea,1238,Processing,15305,Processing,89.779999,2022-01-06 21:34:29,High,Ray-Ban Unisex RB4021P Polarized Sunglasses,Plus,Ray-Ban,89.779999,42.106819


Broadcast Final DataFrame Count: 144100


### Repartitioning and Output

In [34]:
# Repartition final_df by state
final_df_repartitioned = final_df.repartition("state")
print("Number of partitions after repartition:")
print(final_df_repartitioned.rdd.getNumPartitions())

Number of partitions after repartition:
17


In [41]:
# Parquet format
final_df_repartitioned.write.mode("overwrite").parquet(
    "C:/temp/final_df_by_state_parquet"
)

Py4JJavaError: An error occurred while calling o858.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

In [ ]:
# Read back and verify
parquet_df = spark.read.parquet("output/final_df_by_state_parquet")

print("Parquet Row Count:")
print(parquet_df.count())

parquet_df.show(5, truncate=False)

In [43]:

monthly_order_count = final_df.withColumn(
    "order_month",
    date_format(col("item_created_at"), "yyyy-MM")
).groupBy("order_month") \
 .agg(countDistinct("order_id").alias("monthly_order_count")) \
 .orderBy("order_month")

monthly_order_count.show(50, truncate=False)

+-----------+-------------------+
|order_month|monthly_order_count|
+-----------+-------------------+
|2019-01    |25                 |
|2019-02    |82                 |
|2019-03    |151                |
|2019-04    |212                |
|2019-05    |279                |
|2019-06    |369                |
|2019-07    |457                |
|2019-08    |519                |
|2019-09    |580                |
|2019-10    |685                |
|2019-11    |756                |
|2019-12    |848                |
|2020-01    |953                |
|2020-02    |927                |
|2020-03    |1119               |
|2020-04    |1178               |
|2020-05    |1288               |
|2020-06    |1336               |
|2020-07    |1527               |
|2020-08    |1617               |
|2020-09    |1739               |
|2020-10    |1892               |
|2020-11    |2041               |
|2020-12    |2185               |
|2021-01    |2338               |
|2021-02    |2296               |
|2021-03    |2

In [44]:
from pyspark.sql.functions import count, desc, row_number
from pyspark.sql.window import Window

# Count items sold by state and category
state_category_sales = final_df.groupBy("state", "category") \
    .agg(count("*").alias("items_sold"))

# Create window by state and order categories by items sold
state_window = Window.partitionBy("state").orderBy(desc("items_sold"))

# Rank categories within each state
top_3_categories_each_state = state_category_sales.withColumn(
    "rank",
    row_number().over(state_window)
).filter(
    col("rank") <= 3
).orderBy("state", "rank")

top_3_categories_each_state.show(100, truncate=False)

+----------------------------+-----------------------------+----------+----+
|state                       |category                     |items_sold|rank|
+----------------------------+-----------------------------+----------+----+
|Acre                        |Fashion Hoodies & Sweatshirts|6         |1   |
|Acre                        |Swim                         |5         |2   |
|Acre                        |Sweaters                     |4         |3   |
|Aichi                       |Tops & Tees                  |10        |1   |
|Aichi                       |Jeans                        |8         |2   |
|Aichi                       |Sleep & Lounge               |7         |3   |
|Akita                       |Plus                         |1         |1   |
|Alabama                     |Intimates                    |18        |1   |
|Alabama                     |Fashion Hoodies & Sweatshirts|18        |2   |
|Alabama                     |Sweaters                     |17        |3   |

In [46]:
from pyspark.sql.functions import avg, col, desc

category_price_comparison = final_df.withColumn(
    "margin",
    col("sale_price") - col("cost")
).groupBy("category") \
 .agg(
     avg("sale_price").alias("avg_sale_price"),
     avg("retail_price").alias("avg_retail_price"),
     avg("cost").alias("avg_cost"),
     avg("margin").alias("avg_margin")
 ) \
 .orderBy(desc("avg_sale_price"))

#category_price_comparison.show(truncate=False)
display(category_price_comparison.toPandas())

,category,avg_sale_price,avg_retail_price,avg_cost,avg_margin
0,Outerwear & Coats,145.484174,145.484174,64.622909,80.861264
1,Suits & Sport Coats,135.609516,135.609516,54.401471,81.208045
2,Suits,120.716414,120.716414,72.870185,47.846229
3,Blazers & Jackets,117.681776,117.681776,44.625555,73.056221
4,Dresses,100.352951,100.352951,45.322812,55.030139
5,Jeans,100.096204,100.096204,53.553744,46.542460
6,Clothing Sets,90.795506,90.795506,55.763229,35.032277
7,Sweaters,80.988078,80.988078,39.064871,41.923207
8,Jumpsuits & Rompers,72.750531,72.750531,38.836583,33.913948
9,Accessories,64.965004,64.965004,26.000936,38.964068
